# Fase 2 — Descritores Composicionais via Matminer/Magpie

**Data:** 2026-03  
**Depende de:** `data/processed/perovskita.csv` e `kesterita.csv` (Fase 1)  
**Objetivo:** gerar features composicionais Magpie para os candidatos PV e IBSC,  
aplicar seleção por variância e colinearidade, e analisar correlação feature×gap.

## Subconjuntos de trabalho

| Filtro | Critério |
|---|---|
| `usar_em_analise == True` | estrutura confirmada/relacionada + regime DFT conhecido |
| `is_metal == False` | semicondutores e isolantes apenas |
| `energy_above_hull < 0.5 eV/átomo` | remove outliers termodinâmicos extremos |
| Estratificação por `regime_calc` | GGA e GGA+U tratados separadamente |

> **Nota sobre kesteritas:** n=234 (subconjunto de análise ~114 GGA não-metais).  
> Modelos ML na Fase 3 serão mais conservadores para esta família.  
> Gap GGA-PBE subestima severamente — rótulos tratados com cautela.

In [ ]:
# ── Instalação (rodar uma vez por sessão Colab) ───────────────────────────────
# !pip install mp-api pymatgen matminer python-dotenv -q

# ── Colab: injetar chave de API ───────────────────────────────────────────────
# import os
# from google.colab import userdata
# os.environ["MP_API_KEY"] = userdata.get("MP_API_KEY")

# ── Colab: importar extraction.py ────────────────────────────────────────────
# import sys
# !git clone https://github.com/seuusuario/materials-solar-ml.git 2>/dev/null || git -C materials-solar-ml pull
# sys.path.insert(0, "/content/materials-solar-ml/src")

In [ ]:
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Caminhos ──────────────────────────────────────────────────────────────────
PASTA_PROCESSED = "data/processed"          # ajustar se necessário
PASTA_FASE2     = "data/fase2"              # saída desta fase

import os
os.makedirs(PASTA_FASE2, exist_ok=True)

print("Imports OK")

In [ ]:
from extraction import carregar, adicionar_features, PV_GAP_MIN, PV_GAP_MAX, IBSC_GAP_MAX, HULL_THRESH

df_p = carregar("perovskita", pasta=PASTA_PROCESSED)
df_k = carregar("kesterita",  pasta=PASTA_PROCESSED)

print(f"Perovskitas Duplas carregadas: {len(df_p):>5} materiais, {len(df_p.columns)} colunas")
print(f"Kesteritas carregadas:         {len(df_k):>5} materiais, {len(df_k.columns)} colunas")

# Verificar colunas críticas da Fase 1
COLUNAS_CRITICAS = ["usar_em_analise", "estrutura_esperada", "regime_calc",
                    "is_metal", "band_gap", "energy_above_hull"]
for col in COLUNAS_CRITICAS:
    ok = (col in df_p.columns) and (col in df_k.columns)
    print(f"  {'✓' if ok else '✗ AUSENTE'} {col}")

## 1. Definição dos subconjuntos de trabalho

In [ ]:
# ── Filtros base (aplicados a ambas as famílias) ──────────────────────────────
HULL_MAX_ML = 0.50   # eV/átomo — remove outliers termodinâmicos extremos
                     # mais restritivo que HULL_THRESH (0.05) mas menos que sem filtro
                     # justificativa: materiais com hull > 0.5 eV são improváveis de sintetizar
                     # e introduzem ruído nos modelos de gap

def subconjunto_analise(df, hull_max=HULL_MAX_ML):
    """Retorna subconjunto filtrado para análise e ML."""
    mask = (
        df["usar_em_analise"] &
        (~df["is_metal"]) &
        (df["energy_above_hull"] < hull_max)
    )
    return df[mask].copy()

sub_p = subconjunto_analise(df_p)
sub_k = subconjunto_analise(df_k)

# Distribuição por regime dentro de cada subconjunto
for nome, sub in [("Perovskitas Duplas", sub_p), ("Kesteritas", sub_k)]:
    print(f"\n── {nome} (n={len(sub)}) ──")
    print(sub["regime_calc"].value_counts().to_string())
    print(f"  Gap médio:    {sub['band_gap'].mean():.3f} eV")
    print(f"  Gap mediana:  {sub['band_gap'].median():.3f} eV")
    print(f"  Hull médio:   {sub['energy_above_hull'].mean():.3f} eV/át.")

# Candidatos PV e IBSC dentro dos subconjuntos
print("\n── Candidatos (subconjuntos filtrados) ──")
for nome, sub in [("PD", sub_p), ("K", sub_k)]:
    n_pv   = sub["is_pv_candidate"].sum()
    n_ibsc = sub["is_ibsc_candidate"].sum()
    print(f"  {nome}: PV={n_pv}  IBSC={n_ibsc}  (total={len(sub)})")

## 2. Geração de features Magpie

O preset `magpie` do matminer gera 132 features (versão 0.10.0) para cada composição:  
estatísticas (min, max, range, mean, avg_dev, mode) de 22 propriedades atômicas elementares  
(número atômico, eletronegatividade, raio, valência, energia de ionização, etc.).

**Tempo estimado:** ~0.5–2 ms por composição → ~1–5 min para n=2.000–3.000 materiais.

> ⚠️ **Atenção:** features Magpie são estritamente composicionais — não capturam  
> informação estrutural (simetria, distâncias de ligação, ângulos).  
> Para a Fase 3, isso é um limite metodológico a discutir no artigo.

In [ ]:
# ── Inicializar featurizador Magpie ───────────────────────────────────────────
featurizer = ElementProperty.from_preset("magpie")
featurizer.set_n_jobs(1)   # 1 job evita problemas de fork no Colab
NOMES_MAGPIE = featurizer.feature_labels()

print(f"Magpie inicializado: {len(NOMES_MAGPIE)} features")
print(f"Exemplos: {NOMES_MAGPIE[:5]}")
print(f"     ...  {NOMES_MAGPIE[-3:]}")

In [ ]:
def aplicar_magpie(df, featurizer, col_formula="formula", verbose=True):
    """
    Gera features Magpie para cada composição no DataFrame.

    Fluxo:
      1. Converte col_formula → pymatgen.Composition
      2. Aplica featurizer linha-a-linha (mais robusto que featurize_dataframe)
      3. Registra erros sem interromper (material fica com NaN nas features)

    Retorna:
      df_orig com colunas Magpie adicionadas (sem alterar df original)
    """
    nomes = featurizer.feature_labels()
    resultados = []
    erros = []

    for i, row in df.iterrows():
        formula = row[col_formula]
        try:
            comp = Composition(formula)
            feats = featurizer.featurize(comp)
            resultados.append(feats)
        except Exception as e:
            resultados.append([np.nan] * len(nomes))
            erros.append((row.get("material_id", i), formula, str(e)))

        if verbose and (len(resultados) % 200 == 0):
            print(f"  {len(resultados)}/{len(df)} processados...")

    df_feats = pd.DataFrame(resultados, columns=nomes, index=df.index)

    if erros:
        print(f"  ⚠ {len(erros)} erros de featurização:")
        for mid, formula, msg in erros[:5]:
            print(f"    {mid} ({formula}): {msg}")

    return pd.concat([df.reset_index(drop=True),
                      df_feats.reset_index(drop=True)], axis=1)


print("Função aplicar_magpie definida.")
print("\nTeste rápido com Cs2AgBiBr6:")
teste = pd.DataFrame({"formula": ["Cs2AgBiBr6", "Cu2ZnSnS4"],
                      "band_gap": [1.35, 0.09]})
teste_feat = aplicar_magpie(teste, featurizer, verbose=False)
print(f"  Shape resultado: {teste_feat.shape}")
print(f"  NaNs nas features: {teste_feat[NOMES_MAGPIE].isna().sum().sum()}")

In [ ]:
# ── Aplicar Magpie às PD ──────────────────────────────────────────────────────
print(f"Gerando features Magpie para {len(sub_p)} perovskitas duplas...")
sub_p_feat = aplicar_magpie(sub_p, featurizer, verbose=True)

# Verificação rápida
n_nan_p = sub_p_feat[NOMES_MAGPIE].isna().any(axis=1).sum()
print(f"\n✓ PD concluído: {len(sub_p_feat)} materiais, {n_nan_p} com algum NaN")

In [ ]:
# ── Aplicar Magpie às Kesteritas ──────────────────────────────────────────────
print(f"Gerando features Magpie para {len(sub_k)} kesteritas...")
sub_k_feat = aplicar_magpie(sub_k, featurizer, verbose=True)

n_nan_k = sub_k_feat[NOMES_MAGPIE].isna().any(axis=1).sum()
print(f"\n✓ K concluído: {len(sub_k_feat)} materiais, {n_nan_k} com algum NaN")

In [ ]:
# ── Exportar datasets com features ───────────────────────────────────────────
path_p = os.path.join(PASTA_FASE2, "perovskita_magpie.csv")
path_k = os.path.join(PASTA_FASE2, "kesterita_magpie.csv")

sub_p_feat.to_csv(path_p, index=False)
sub_k_feat.to_csv(path_k, index=False)

print(f"Exportado: {path_p}  ({len(sub_p_feat)} linhas, {len(sub_p_feat.columns)} colunas)")
print(f"Exportado: {path_k}  ({len(sub_k_feat)} linhas, {len(sub_k_feat.columns)} colunas)")

## 3. Seleção de features

Três critérios aplicados sequencialmente:

1. **Variância zero ou quase zero** — features constantes ou quase-constantes para  
   uma família têm poder preditivo nulo e adicionam ruído.
2. **Colinearidade alta** (r > 0.95) — features redundantes inflariam a importância  
   de grupos de descritores correlacionados. Remove a segunda de cada par.
3. **Correlação com o target** (opcional, exploratório) — top features correlacionadas  
   com `band_gap` como filtro de relevância.

In [ ]:
def selecionar_features(
    df, nomes_features, target="band_gap",
    limiar_var=0.01, limiar_corr=0.95,
    verbose=True
):
    """
    Seleciona features Magpie em 3 etapas:
      1. Remove variância quase-zero (std < limiar_var)
      2. Remove colineares (r > limiar_corr) — mantém primeira do par
      3. Retorna lista ordenada por correlação absoluta com o target

    Retorna:
      (features_selecionadas, dict_relatorio)
    """
    X = df[nomes_features].copy()
    n_inicial = len(nomes_features)

    # ── Passo 1: variância ────────────────────────────────────────────────────
    std = X.std()
    baixa_var = std[std < limiar_var].index.tolist()
    X = X.drop(columns=baixa_var)
    if verbose:
        print(f"  Passo 1 (variância < {limiar_var}): removidas {len(baixa_var)} features")
        print(f"           Restantes: {X.shape[1]}")

    # ── Passo 2: colinearidade ────────────────────────────────────────────────
    # Matriz de correlação absoluta; percorre triangular superior
    corr_mat = X.corr().abs()
    upper = corr_mat.where(
        np.triu(np.ones(corr_mat.shape), k=1).astype(bool)
    )
    remover_colinear = [col for col in upper.columns
                        if any(upper[col] > limiar_corr)]
    X = X.drop(columns=remover_colinear)
    if verbose:
        print(f"  Passo 2 (|r| > {limiar_corr}):      removidas {len(remover_colinear)} features")
        print(f"           Restantes: {X.shape[1]}")

    # ── Passo 3: correlação com target ────────────────────────────────────────
    if target in df.columns:
        y = df[target].dropna()
        idx_comum = X.index.intersection(y.index)
        corr_target = X.loc[idx_comum].corrwith(y.loc[idx_comum]).abs().sort_values(ascending=False)
        top10 = corr_target.head(10)
        if verbose:
            print(f"\n  Top 10 features por |corr| com {target}:")
            for feat, val in top10.items():
                print(f"    {feat:<45} {val:.3f}")
    else:
        corr_target = None

    relatorio = {
        "n_inicial": n_inicial,
        "removidas_var": len(baixa_var),
        "removidas_colinear": len(remover_colinear),
        "n_final": X.shape[1],
        "nomes_baixa_var": baixa_var,
        "nomes_colinear": remover_colinear,
        "corr_target": corr_target,
    }
    return X.columns.tolist(), relatorio


print("Função selecionar_features definida.")

In [ ]:
print("══ Seleção de features — Perovskitas Duplas ══")
feats_p, rel_p = selecionar_features(
    sub_p_feat, NOMES_MAGPIE, target="band_gap", verbose=True
)
print(f"\n  RESULTADO: {rel_p['n_inicial']} → {rel_p['n_final']} features selecionadas")

In [ ]:
print("══ Seleção de features — Kesteritas ══")
feats_k, rel_k = selecionar_features(
    sub_k_feat, NOMES_MAGPIE, target="band_gap", verbose=True
)
print(f"\n  RESULTADO: {rel_k['n_inicial']} → {rel_k['n_final']} features selecionadas")

# ── Interseção e diferença entre famílias ─────────────────────────────────────
set_p = set(feats_p)
set_k = set(feats_k)
print(f"\nFeatures em comum:          {len(set_p & set_k)}")
print(f"Exclusivas PD:              {len(set_p - set_k)}")
print(f"Exclusivas Kesteritas:      {len(set_k - set_p)}")

# Features selecionadas para análise conjunta (intersecção)
feats_comuns = sorted(set_p & set_k)
print(f"\nFeatures comuns (base para comparação inter-família): {len(feats_comuns)}")

## 4. Análise de correlação feature × gap

Comparação sistemática de como cada feature Magpie se correlaciona com `band_gap`  
nas duas famílias — diferenças relevantes têm interpretação física e são candidatas  
a discussão no artigo.

In [ ]:
def tabela_corr_comparativa(df_p, df_k, features, target="band_gap", top_n=20):
    """
    Retorna DataFrame com correlação Pearson de cada feature com o target,
    para PD e Kesteritas, ordenado por |corr_PD|.
    """
    corr_p = df_p[features + [target]].corr()[target].drop(target)
    corr_k = df_k[features + [target]].corr()[target].drop(target)

    comp = pd.DataFrame({
        "corr_PD":        corr_p,
        "corr_K":         corr_k,
        "|corr_PD|":      corr_p.abs(),
        "|corr_K|":       corr_k.abs(),
        "sinal_igual":    corr_p.sign() == corr_k.sign(),
    }).sort_values("|corr_PD|", ascending=False)

    return comp.head(top_n)


print("── Top 20 features por correlação com band_gap (PD) ──")
tab_corr = tabela_corr_comparativa(sub_p_feat, sub_k_feat, feats_comuns)
print(tab_corr[["corr_PD", "corr_K", "sinal_igual"]].to_string())

In [ ]:
# ── Identificar features com sinal oposto (física potencialmente distinta) ────
tab_completa = tabela_corr_comparativa(
    sub_p_feat, sub_k_feat, feats_comuns, top_n=len(feats_comuns)
)
sinal_oposto = tab_completa[
    (~tab_completa["sinal_igual"]) &
    (tab_completa["|corr_PD|"] > 0.10) &
    (tab_completa["|corr_K|"]  > 0.10)
]
print(f"Features com sinal de correlação oposto entre famílias (|r|>0.10 em ambas):")
print(sinal_oposto[["corr_PD", "corr_K"]].to_string())

In [ ]:
# ── Fig. 7 — Heatmap de correlação Magpie × gap (Top N features) ─────────────
# Usar top 20 features por |corr_PD| para manter legibilidade
TOP_N_PLOT = 20
top_feats = tab_corr.head(TOP_N_PLOT).index.tolist()

# Nomes abreviados para o eixo Y (remover prefixo 'MagpieData ')
nomes_abrev = [f.replace("MagpieData ", "") for f in top_feats]

corr_p_top = sub_p_feat[top_feats + ["band_gap"]].corr()["band_gap"].drop("band_gap").values
corr_k_top = sub_k_feat[top_feats + ["band_gap"]].corr()["band_gap"].drop("band_gap").values

data_plot = pd.DataFrame(
    {"Perovskitas Duplas": corr_p_top, "Kesteritas": corr_k_top},
    index=nomes_abrev
)

fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(
    data_plot, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={"size": 9}
)
ax.set_title("Correlação Pearson: features Magpie × band gap (DFT)",
             fontsize=12, pad=12)
ax.set_xlabel("Família", fontsize=11)
ax.set_ylabel("Feature Magpie", fontsize=11)
plt.tight_layout()
plt.savefig("fig7_corr_magpie_gap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: fig7_corr_magpie_gap.png")

## 5. Features Magpie × candidatos PV/IBSC

Comparar distribuições de features-chave entre candidatos e não-candidatos.  
Candidatos com perfil composicional distinto → feature tem poder discriminativo  
além da correlação com o gap contínuo.

In [ ]:
def plot_features_candidatos(df, features_top, familia_nome, janela="pv", n_feats=6):
    """
    Box-plots das top N features comparando candidatos vs não-candidatos.

    janela: 'pv'  → is_pv_candidate
            'ibsc' → is_ibsc_candidate
    """
    col_cand = "is_pv_candidate" if janela == "pv" else "is_ibsc_candidate"
    label    = "PV" if janela == "pv" else "IBSC"

    df_plot = df.copy()
    df_plot["grupo"] = df_plot[col_cand].map({True: f"Candidato {label}", False: "Outros"})

    feats_plot = [f.replace("MagpieData ", "") for f in features_top[:n_feats]]
    df_plot = df_plot.rename(columns={f: f.replace("MagpieData ", "") for f in features_top})

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle(f"{familia_nome} — Features Magpie: candidatos {label} vs outros",
                 fontsize=12, fontweight="bold")
    axes = axes.flatten()

    for i, feat in enumerate(feats_plot):
        if feat not in df_plot.columns:
            continue
        sns.boxplot(
            data=df_plot, x="grupo", y=feat, ax=axes[i],
            palette={f"Candidato {label}": "#2196F3", "Outros": "#9E9E9E"},
            showfliers=False
        )
        axes[i].set_title(feat, fontsize=10)
        axes[i].set_xlabel("")
        axes[i].tick_params(axis="x", rotation=15)

    plt.tight_layout()
    fname = f"fig8_{familia_nome.lower().replace(' ', '_')}_{janela}_magpie.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvo: {fname}")


# Top 6 features por correlação com gap (PD)
top6_p = tab_corr.head(6).index.tolist()

plot_features_candidatos(sub_p_feat, top6_p, "Perovskitas Duplas", janela="pv")
plot_features_candidatos(sub_p_feat, top6_p, "Perovskitas Duplas", janela="ibsc")

In [ ]:
# ── Para kesteritas — usar features por correlação com gap das kesteritas ─────
top6_k = (
    sub_k_feat[feats_comuns + ["band_gap"]].corr()["band_gap"]
    .drop("band_gap").abs().sort_values(ascending=False).head(6).index.tolist()
)

plot_features_candidatos(sub_k_feat, top6_k, "Kesteritas", janela="pv")

## 6. Estratificação por regime de cálculo

Conforme documentado na Fase 1, comparação direta GGA vs GGA+U é metodologicamente  
incorreta. Esta seção verifica se as features Magpie têm distribuições similares  
entre regimes — se sim, o dataset combinado pode ser usado com a variável de regime  
como feature adicional; se não, modelos separados são necessários.

In [ ]:
# ── Teste estatístico simples: diferença de média das top features por regime ─
from scipy import stats

print("── Diferença das top features entre GGA e GGA+U (PD) ──")
gga   = sub_p_feat[sub_p_feat["regime_calc"] == "GGA"]
ggaU  = sub_p_feat[sub_p_feat["regime_calc"] == "GGA+U"]

resultados_kruskal = []
for feat in top6_p:
    nome = feat.replace("MagpieData ", "")
    a = gga[feat].dropna()
    b = ggaU[feat].dropna()
    stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    resultados_kruskal.append({
        "feature": nome,
        "media_GGA":  round(a.mean(), 3),
        "media_GGA+U": round(b.mean(), 3),
        "p_valor": round(p, 4),
        "diferenca_sig": p < 0.05
    })

df_kruskal = pd.DataFrame(resultados_kruskal).set_index("feature")
print(df_kruskal.to_string())
print()
n_sig = df_kruskal["diferenca_sig"].sum()
print(f"Features com distribuição significativamente diferente entre regimes: {n_sig}/{len(top6_p)}")
print()
if n_sig > len(top6_p) // 2:
    print("  → Atenção: maioria das features difere entre regimes.")
    print("    Considerar incluir regime_calc como feature categórica no modelo,")
    print("    ou treinar modelos separados por regime.")
else:
    print("  → Features relativamente estáveis entre regimes.")
    print("    Modelo conjunto com regime como feature é razoável.")

## 7. Dataset final para Fase 3 (ML)

Exporta datasets prontos para modelagem, com:
- Features Magpie selecionadas
- Features da Fase 1 relevantes para ML (density, site_density, efermi)
- Target: `band_gap`
- Metadados de rastreabilidade: material_id, formula, regime_calc, estrutura_esperada

In [ ]:
# ── Features da Fase 1 a incluir ──────────────────────────────────────────────
# Selecionadas com base nos achados do mapa de correlação da Fase 1
FEATS_FASE1 = [
    "density",         # mais informativa para K que para PD
    "site_density",    # compacidade estrutural
    "efermi",          # correlação moderada com gap nas K
    "volume",          # manter para PD (correlação moderada)
    "nsites",          # ATENÇÃO: colinear com volume nas K (r=0.91)
                       # será dropado na Fase 3 para kesteritas se necessário
]
FEATS_FASE1_DISPONIVEIS_P = [f for f in FEATS_FASE1 if f in sub_p_feat.columns]
FEATS_FASE1_DISPONIVEIS_K = [f for f in FEATS_FASE1 if f in sub_k_feat.columns]

# ── Colunas de metadados e target ─────────────────────────────────────────────
METADADOS = ["material_id", "formula", "regime_calc", "estrutura_esperada",
             "is_pv_candidate", "is_ibsc_candidate",
             "is_pv_candidate_amp", "is_ibsc_candidate_amp",
             "energy_above_hull", "is_stable", "near_hull",
             "is_gap_direct", "ordering"]
TARGET = ["band_gap"]


def preparar_dataset_ml(df_feat, feats_magpie, feats_fase1, metadados, target, nome):
    """Monta e exporta dataset ML final."""
    cols_meta_disp    = [c for c in metadados if c in df_feat.columns]
    cols_fase1_disp   = [c for c in feats_fase1 if c in df_feat.columns]
    cols_magpie_disp  = [c for c in feats_magpie if c in df_feat.columns]

    df_ml = df_feat[cols_meta_disp + target + cols_fase1_disp + cols_magpie_disp].copy()

    # Codificar regime_calc como dummy (para modelos que não aceitam categórico)
    df_ml["regime_GGA"]   = (df_ml["regime_calc"] == "GGA").astype(int)
    df_ml["regime_GGAU"]  = (df_ml["regime_calc"] == "GGA+U").astype(int)
    df_ml["regime_HSE06"] = (df_ml["regime_calc"] == "HSE06").astype(int)

    # Codificar gap_direct
    if "is_gap_direct" in df_ml.columns:
        df_ml["gap_direto"] = df_ml["is_gap_direct"].astype(int)

    n_nan = df_ml[cols_magpie_disp].isna().any(axis=1).sum()

    path = os.path.join(PASTA_FASE2, f"{nome}_ml_ready.csv")
    df_ml.to_csv(path, index=False)

    print(f"\n── {nome.upper()} — dataset ML pronto ──")
    print(f"  Linhas:            {len(df_ml)}")
    print(f"  Features Magpie:   {len(cols_magpie_disp)}")
    print(f"  Features Fase 1:   {len(cols_fase1_disp)}")
    print(f"  Materiais c/ NaN:  {n_nan}")
    print(f"  Exportado:         {path}")
    return df_ml


ml_p = preparar_dataset_ml(
    sub_p_feat, feats_p, FEATS_FASE1_DISPONIVEIS_P, METADADOS, TARGET, "perovskita"
)
ml_k = preparar_dataset_ml(
    sub_k_feat, feats_k, FEATS_FASE1_DISPONIVEIS_K, METADADOS, TARGET, "kesterita"
)

In [ ]:
# ── Resumo final dos datasets para Fase 3 ────────────────────────────────────
print("══ RESUMO — Datasets prontos para Fase 3 (ML) ══")
print()

for nome, ml, feats in [("Perovskitas Duplas", ml_p, feats_p),
                         ("Kesteritas",         ml_k, feats_k)]:
    n_total  = len(ml)
    n_pv     = ml["is_pv_candidate"].sum()
    n_ibsc   = ml["is_ibsc_candidate"].sum()
    n_feats  = len(feats) + len(FEATS_FASE1)
    n_gga    = (ml["regime_calc"] == "GGA").sum()
    n_ggaU   = (ml["regime_calc"] == "GGA+U").sum()
    n_hse    = (ml["regime_calc"] == "HSE06").sum()
    n_nan    = ml[feats].isna().any(axis=1).sum()

    print(f"{nome}")
    print(f"  n total:         {n_total}")
    print(f"  Candidatos PV:   {n_pv}  |  IBSC: {n_ibsc}")
    print(f"  Regimes:         GGA={n_gga}  GGA+U={n_ggaU}  HSE06={n_hse}")
    print(f"  Features total:  ~{n_feats} (Magpie selecionadas + Fase 1 + dummies regime)")
    print(f"  Linhas com NaN:  {n_nan}")
    print()

print("Próximos passos (Fase 3):")
print("  1. Baseline: LinearRegression / Ridge com todas as features selecionadas")
print("  2. Modelo principal: RandomForest ou XGBoost com CV estratificado por regime")
print("  3. SHAP values para interpretação — confirmar física das features importantes")
print("  4. Análise separada GGA vs GGA+U se o teste de regime desta fase indicar necessidade")

## Decisões metodológicas — registro para o artigo

| Decisão | Escolha | Justificativa |
|---|---|---|
| Preset Magpie | Completo (132 features) | Reprodutível, sem seleção a priori subjetiva |
| Seleção variância | std < 0.01 | Remove constantes e quase-constantes |
| Seleção colinearidade | \|r\| > 0.95 | Mantém primeira de cada par redundante |
| Features Fase 1 incluídas | density, site_density, efermi, volume, nsites | Achados do mapa de correlação da Fase 1 |
| Regime como feature | Dummies GGA/GGA+U/HSE06 | Permite modelo conjunto com controle de funcional |
| Hull máximo no subconjunto ML | 0.50 eV/átomo | Remove outliers termodinâmicos extremos |
| Kesteritas: gap subestimado | Documentado como limitação | Correção HSE06 prevista para top candidatos na Fase 4 |

> **Sobre o número de features Magpie:**  
> A versão 0.10.0 do matminer retorna 132 features (não 114 como em versões anteriores).  
> A diferença decorre de propriedades atômicas adicionadas nas versões recentes do preset.  
> Registrar a versão exata do matminer no `requirements.txt` para reprodutibilidade.